In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

secret = os.getenv("SECRET")
id = os.getenv("ID")


In [3]:
from osu import Client, GameModeStr
import requests

client = Client.from_credentials(id, secret, None)
res = client.search_beatmapsets(filters={"m": GameModeStr.STANDARD})

import time
from requests.exceptions import ConnectionError

for i in range(1, 21):
    while True:  # retry until success
        try:
            res.beatmapsets.extend(
                client.search_beatmapsets(
                    filters={"m": GameModeStr.STANDARD}, page=i
                ).beatmapsets
            )
            print(f"Finished page: {i}")
            break  # success, exit retry loop
        except ConnectionError:
            print(f"Connection error on page {i}, retrying in 5s...")
            time.sleep(5)


Finished page: 1
Finished page: 2
Finished page: 3
Finished page: 4
Finished page: 5
Finished page: 6
Finished page: 7
Finished page: 8
Finished page: 9
Finished page: 10
Finished page: 11
Finished page: 12
Finished page: 13
Finished page: 14
Finished page: 15
Finished page: 16
Finished page: 17
Finished page: 18
Finished page: 19
Finished page: 20


In [28]:
import os
import json

existing_ids = []
for db_fpaths in [os.path.join("../db", fn) for fn in os.listdir("../db") if fn != "info.json"]:
    with open(db_fpaths, "r", encoding="utf-8") as db_fp:
        existing_ids.extend([item["id"] for item in json.load(db_fp)])
        db_fp.close()
print(f"Number of existing beatmaps: {len(existing_ids)}")

beatmap_ids = []
for item in res.beatmapsets:
    for b in item.beatmaps:
        if 4 <= b.difficulty_rating and not b.id in existing_ids:
            beatmap_ids.append((b.id, b.beatmapset_id, item.tags, item.last_updated.isoformat()[:10]))

print(f"Number of new (unique) beatmaps: {len(beatmap_ids)}")


Number of existing beatmaps: 53979
Number of new (unique) beatmaps: 63


In [4]:
tmpv = 0
new_files = []
existing_files = os.listdir("./osu-files")
for b_id,beatmapset_id, tags, last_updated in beatmap_ids:
    while True:
        try:
            new_fn = f"{b_id}_{beatmapset_id}.osu"
            new_fpath = f"osu-files/{b_id}_{beatmapset_id}.osu"
            if new_fn in existing_files:
                tmpv += 1
                print(f"File: {new_fn} already exists! {tmpv}/{len(beatmap_ids)}")
                break
            osu_data = requests.get(f"https://osu.ppy.sh/osu/{b_id}").content.decode("utf-8-sig").splitlines()
            with open(new_fpath, "w", encoding="utf-8") as fp:
                fp.writelines(l + "\n" for l in osu_data)
                fp.close()
                tmpv += 1
                print(f"{tmpv}/{len(beatmap_ids)}")
                new_files.append((f"osu-files/{b_id}_{beatmapset_id}.osu", b_id, beatmapset_id))
                break
        except ConnectionError:
            print("Connection error. Retrying after 60s...")
            time.sleep(60)


File: 4984117_79175.osu already exists! 1/2824
File: 5435314_2478116.osu already exists! 2/2824
File: 4776138_2246482.osu already exists! 3/2824
File: 4789911_2246482.osu already exists! 4/2824
File: 4814342_2246482.osu already exists! 5/2824
File: 4985474_2246482.osu already exists! 6/2824
File: 5468216_2489622.osu already exists! 7/2824
File: 5468218_2489622.osu already exists! 8/2824
File: 5468219_2489622.osu already exists! 9/2824
File: 5471100_2490629.osu already exists! 10/2824
File: 5471102_2490629.osu already exists! 11/2824
File: 5471103_2490629.osu already exists! 12/2824
File: 5360450_2453440.osu already exists! 13/2824
File: 5442555_2480769.osu already exists! 14/2824
File: 5462275_2480769.osu already exists! 15/2824
File: 5463989_2480769.osu already exists! 16/2824
File: 5481097_2494107.osu already exists! 17/2824
File: 5481099_2494107.osu already exists! 18/2824
File: 4592878_2175201.osu already exists! 19/2824
File: 4672266_2175201.osu already exists! 20/2824
File: 52155

In [14]:
to_executor = [(f"osu-files/{id}_{beatmapset_id}.osu", beatmapset_id, f"https://assets.ppy.sh/beatmaps/{beatmapset_id}/covers/cover.jpg", f"https://osu.ppy.sh/beatmapsets/{beatmapset_id}#osu/{id}", tags, last_updated) for id, beatmapset_id, tags, last_updated in beatmap_ids]
to_executor[0]

('osu-files/4984117_79175.osu',
 79175,
 'https://assets.ppy.sh/beatmaps/79175/covers/cover.jpg',
 'https://osu.ppy.sh/beatmapsets/79175#osu/4984117',
 'hugo pierre leclercq alphabeat bag raiders black eyed peas britney spears capsule chromeo coldplay daft punk deadmau5 ellie goulding elo electric light orchestra girls aloud gorillaz gossip gwen stefani housse racket justice katy perry kesha kylie minogue lady gaga linkin park madonna marting solveig dragonette michael jackson nero one-t one republic ratatat solange stardust buggles killers the who yelle remix mashup english electronic dance music edm live gabe elbow_sniffer elbow sniffer ohczarr lune raburauza',
 '2026-01-16')

In [ ]:
if __name__ == "__main__":
    import os
    from concurrent.futures import ProcessPoolExecutor, as_completed
    from osu_file_parser import create_stats_entry

    fpaths = [os.path.join("osu-files", fn) for fn in os.listdir("osu-files")]
    calculated = []
    
    with ProcessPoolExecutor(max_workers=5) as executor:
        futures = [executor.submit(create_stats_entry, f, beatmapset_id, bg_url, url, tags, last_updated, True) for f, beatmapset_id, bg_url, url, tags, last_updated in to_executor]
        for fut in as_completed(futures):
            stats = fut.result()
            if isinstance(stats, dict):
                calculated.append(fut.result())

    print(f"length calculated: {len(calculated)}")


length calculated: 2783


In [ ]:
import json
from datetime import datetime

now = datetime.now()
with open(f"../db/update-{now.year}-{now.month}-{now.day}.json", "w") as fp:
    json.dump(calculated, fp, indent=2)
    print(f"new update dropped! {now.year}-{now.month}-{now.day}")


new update dropped! 2026-1-25


In [58]:
### MUST ALWAYS RUN ###
import os
import json

path_to_db = './db'

with open(f"../db/info.json", "w", encoding="utf-8") as fp:
    urls = []
    for fn in os.listdir(f"../db"):
        if "info" in fn:
            continue
        urls.append(f"{path_to_db}/{fn}")
    json.dump(urls, fp, indent=2)
urls


['./db/0-mk88__-lazer.json',
 './db/1-mk88__-lazer.json',
 './db/10-mk88__-lazer.json',
 './db/2-mk88__-lazer.json',
 './db/3-mk88__-lazer.json',
 './db/4-mk88__-lazer.json',
 './db/5-mk88__-lazer.json',
 './db/6-mk88__-lazer.json',
 './db/7-mk88__-lazer.json',
 './db/8-mk88__-lazer.json',
 './db/9-mk88__-lazer.json',
 './db/update-2026-1-25.json']